# Processing Template
___
Jonathan Zhang

Here, we outline how to use this notebook to process stitched and background-subtracted images from HT-BAM experiments. Before you start, carefully read the guidelines below.

## General Notes

### Metadata framework
There exist two sources of metadata to bridge the gap between raw pixels and experimental variables:
- `imaging.csv`: This file logs hardware state and raster metadata for all images taken in an experiment. All information needed to stitch and background subtract images is found here. You may manually overwrite this file to alter how images are processed, but always be sure to create a backup copy beforehand.
    - For the sake of continuity, the stitching and background subtraction modules carry over this data into `stitched_images.csv` and `bgsub_images.csv`.
- `series_index.json`: Metadata specific to the experimental conditions (e.g., titration concentrations) is saved within individual series directories. This ensures that raw data remain linked to their biochemical context. This file is largely irrelevant during stitching, but become important at the processing stage.

### Before you begin
- Manually delete any images you do not want processed beforehand.
- Previous versions of the stitching code required manually directory reorginization. In this version, reorganizing the directory structure is entirely unnecessary and will likely cause the code to fail.
    - Do not rename, move, or create new files within directories of raw images or outputs from the stitching and background subtraction modules (i.e. `raw_images`, `stitched_images`, and `bgsub_images`).
- If you are running processing on a laptop, probably a good idea to close all uneeded applications (code is both memory and CPU intensive).


In [12]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [13]:
from mercury.stitching import ImageStitcher, BackgroundSubtractor
from mercury.processing import Processor
from mercury.processing.experiment import Experiment, Device, DataHandler

from pathlib import Path
import pandas as pd
import numpy as np

# to mute annoying numpy warnings
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

## Image Stitching
The stitching process is designed to be automated and data-driven:
- `stitch_images`: Once the root path is provided, this function acts as the primary orchestrator. It uses the raster parameters defined in `imaging.csv` to find and stitch all raw data associated with the experiment. There are three arguments:
    1. `root`: path to the folder containing experiment data.
    2. `acqui_ori`: specifies corner of the device where imaging starts. In our experiments, this is always from the top-right (i.e. `(True, False)`) 
    3. [OPTIONAL] `rotation`: global rotation parameter. This can drift over time and may need to be adjusted. If stitched images look jagged, this is the first thing that should be tuned. If not provided, rotation will be inferred automatically.
- `stitch_single_raster`: This is the lower-level function called by the main stitcher. If your file-naming convention or folder structure changes substantially from convention, you can bypass the automated discovery logic implemented in `stitch_images` and use this function directly by providing your own path-finding logic.

NOTE: by default, stitched images are not flat-field corrected. If one desires to implement correction for all or a subset of images, one can "hack" the ImageStitcher by manually updating the `apply_ff_correction` column in the `raster_data` attribute of the `ImageStitcher`. For example:

```python
# configure ff correction for just button quant images
bq_mask = stitcher.raster_data['image_path'].apply(str).str.contains('2026-03-09_15-16-09_d2_aura_cyan_2_5_sensitivity_2x2_1_button_quant')
stitcher.raster_data.loc[bq_mask, 'apply_ff_correction'] = True
```

In [14]:
Path.cwd()

PosixPath('/home/freitas/workspace/htbam_may_merge/example_data/garrison/garrison_notebooks_forrelease')

In [ ]:
# init stitcher
root = Path('../20240628')
stitcher = ImageStitcher(root)


Calculating rotation is now done automatically. Still want to test it yourself? Use the following function:

In [6]:
# # OPTIONAL: test different rotation parameters to find the ideal one 
# stitcher.test_stitching_rotations(
#     raster_path = root/'raw_images/2026-05-09_16-58-08_d3_spectra_cyan_2_5_sensitivity_2x2_1_pre_button_quant_brightfield',
#     rotations = [2.2, 1.8, 1.6, 1.4, 1.2, 1.25, 1.0],
#     acqui_origin = (True, False),
#     outdir = root / 'stitch_test'
# )

In [7]:
# stitch the images
stitcher.stitch_images( rotation=None,                  # Automatically find rotation.
                        get_rotation_from='*brightfield*',
                        overlap=None,
                        rotation_method='overlaps',
                        get_overlap_from='*brightfield*',
                        acqui_origin=(True, False))

Auto-detecting optimal rotation from raster: ../20240628/raw_images/20240628-130329-d1_brightfield_2x2_sensitivity_4x/2/20240628-130329_brightfield_2x2_sensitivity_4x_spectra_cyan_2 using method: overlaps
Detected rotation for 20240628-130329_brightfield_2x2_sensitivity_4x_spectra_cyan_2: -0.796 degrees
Using optimal rotation: -0.796 degrees from matching rasters
Auto-detecting optimal overlap from raster: ../20240628/raw_images/20240628-130329-d1_brightfield_2x2_sensitivity_4x/2/20240628-130329_brightfield_2x2_sensitivity_4x_spectra_cyan_2
Detected overlap for 20240628-130329_brightfield_2x2_sensitivity_4x_spectra_cyan_2: 0.1000
Using optimal overlap: 0.1000 from matching rasters


Stitching images.:   1%|          | 2/197 [00:10<17:47,  5.47s/it]

Failed stitching of ../20240628/raw_images/20240628-130429-d1_button_background_multiple_exposures_2x2_sensitivity_4x/2/20240628-130429_button_background_multiple_exposures_2x2_sensitivity_4x_spectra_cyan_2: 


Stitching images.:   3%|▎         | 5/197 [00:21<13:51,  4.33s/it]

Failed stitching of ../20240628/raw_images/20240628-152652-d1_button_quant_multiple_exposures_2x2_sensitivity_4x/2/20240628-152652_button_quant_multiple_exposures_2x2_sensitivity_4x_spectra_cyan_2: 


Stitching images.:   3%|▎         | 6/197 [00:26<14:00,  4.40s/it]


KeyboardInterrupt: 

## Background Subtraction

Like stitching, background subtraction logic is split into two layers to balance automation with developer flexibility:
- `background_subtract_images`: The main function for background subtraction. It automatically groups images based on specified metadata headers (like channel or exposure_time) and applies the subtraction across the entire experiment. It uses `stitched_images.csv` as the source for metadata.
- `background_subtract`: The lower-level function, called within the above, which performs the background subtraction operation on a single image or array. If your file structure changes substantially or you need to implement custom subtraction logic, you can call this function directly and bypass the automated grouping.

### Note on Image Grouping
Users can specify metadata headers in `stitched_images.csv` to use for image grouping (i.e. exposure time, lightsource, etc.). If complex groupings are required, one can manually write new columns to `stitched_images.csv` to define custom logic (but always save a backup copy beforehand).

In [ ]:
# paths to all background images
# use paths relative to "root" variable

background_images = [
    root / 'stitched_images/20240628-130410-d1_button_background_5_2x2_sensitivity_4x/2/20240628-130410_button_background_5_2x2_sensitivity_4x_spectra_cyan_2.tif',
    root / 'stitched_images/20240628-154051-d1_NADPH_background_standard_2x2_dynamic_range_4x_super_uv/5/20240628-154051_NADPH_background_standard_2x2_dynamic_range_4x_super_uv_retra_cyan_5.tif',
    root / 'stitched_images/20240628-162735-d1_NADPH_background_kinetics_1_2x2_dynamic_range_4x_super_uv/5/20240628-162735_NADPH_background_kinetics_1_2x2_dynamic_range_4x_super_uv_retra_cyan_5.tif',
]

settings_to_match = ['temp', 'hum','setup', 'dname', 'lightsource', 'channel', 'exposure', 'camera_mode', 'binning', 'nosepiece', 'apply_ff_correction']
subtractor = BackgroundSubtractor(root)
subtractor.subtract(
    background_images=background_images,
    settings_to_match=settings_to_match,
)

Attempting background subtracting on images with the following settings:
temp: nan | hum: nan | setup: Pinney_Setup_1 | dname: d1 | lightsource: retra_cyan | channel: 5 | exposure: 500 | camera_mode: dynamic_range | binning: 2x2 | nosepiece: 4 | apply_ff_correction: False
Using ../20240628/stitched_images/20240628-154051-d1_NADPH_background_standard_2x2_dynamic_range_4x_super_uv/5/20240628-154051_NADPH_background_standard_2x2_dynamic_range_4x_super_uv_retra_cyan_5.tif as background image


Running background subtraction.: 100%|██████████| 188/188 [02:18<00:00,  1.36it/s]


188 / 188 images were successfully background subtracted

Attempting background subtracting on images with the following settings:
temp: nan | hum: nan | setup: Pinney_Setup_1 | dname: d1 | lightsource: spectra_cyan | channel: 2 | exposure: 5 | camera_mode: sensitivity | binning: 2x2 | nosepiece: 4 | apply_ff_correction: False
Using ../20240628/stitched_images/20240628-130410-d1_button_background_5_2x2_sensitivity_4x/2/20240628-130410_button_background_5_2x2_sensitivity_4x_spectra_cyan_2.tif as background image


Running background subtraction.: 100%|██████████| 3/3 [00:02<00:00,  1.35it/s]

3 / 3 images were successfully background subtracted



## Processing

### Initialization

To begin, initialize an `Experiment` object by providing the root directory of your data. You must then create `Device` objects, representing the devices used within an experiment, and then register them. 

When creating device objects, one can specify pinlists using the `set_pinlist` method. There are two kwargs that are helpful here:
    1. `block_descriptions`- a dictionary specifying which variants are present in each block (for flow-on experiments only)
    2. `pinlist_path`- path to a pinlist file output by `Array-Print` (for expression experiments only) 

For image and metadata management, we use the `DataHandler` class. This class is responsible for fetching images and associated metadata to be processed.

### Processing logic

The execution logic is as follows:
1. Retrieve image data and metadata to be processed via methods implemented in the `DataHandler`.
    - This is done by providing image identifier(s) to the `get_images()` method within the `DataHandler` class.
2. Initialize a processor object.
    - In addition to your experiment object and image data from (1), you will need to designate the type of feature to process (i.e. 'button', 'chamber', or 'all'). 
3. Optionally, set reference images for each device.
    - You'll need to designate the coordinates of the center of all four corner buttons (see below).
4. Process images using the core `process()` method.
    - If you don't have a reference image, pass a single set of coordinates OR a list of N coordinates (N being the number of images to be processed).

Poor feature finding results arise from two problems (almost all the time):
1. Jagged stitches. This can be fixed by going back to stitching and optimizing the rotation parameter.
2. Bad corner picks. Pick new corners.

### A note on corner picking

We have a command line tool for picking corners. Within your terminal, run the following command (be sure to activate the correct conda environment beforehand):

```bash
pick-corners /path/to/stitched/and/subtracted/image.tif
```

In [9]:
# initialize experiment object
experiment = Experiment(root)

# initialize device(s)
# NOTE: dname NEEDS to exactly match that used within the experiment
d1 = Device(setup='s1', dname='d1', dims=(32, 56))
d1.set_pinlist(pinlist_path=root / 'pinlist.csv')

d2 = Device(setup='s1', dname='d2', dims=(32, 56))
d2.set_pinlist(pinlist_path=root / 'pinlist.csv')

# add devices to experiment
experiment.add_device(d1)
experiment.add_device(d2)

# init data handler object
# will print out identifiers that can be passed into the get_images method
data_handler = DataHandler(root)

Loaded the following image sets:
20240628_163448_kinetic_series_adk_adp
20240628_154552_standard_curve_NADPH

Loaded the following images:
20240628-154552_d1_0uM_NADPH_in9_1_retra_cyan_5
20240628-155000_d1_15_625uM_NADPH_bBSA2_1_retra_cyan_5
20240628-155410_d1_31_25uM_NADPH_na3_1_retra_cyan_5
20240628-155820_d1_62_5uM_NADPH_in4_1_retra_cyan_5
20240628-160228_d1_125uM_NADPH_in5_1_retra_cyan_5
20240628-160636_d1_250uM_NADPH_in6_1_retra_cyan_5
20240628-161044_d1_500uM_NADPH_in7_1_retra_cyan_5
20240628-161452_d1_1000uM_NADPH_in8_1_retra_cyan_5
20240628-163448_15_625uM_ADP_retra_cyan_5
20240628-163518_15_625uM_ADP_retra_cyan_5
20240628-163548_15_625uM_ADP_retra_cyan_5
20240628-163618_15_625uM_ADP_retra_cyan_5
20240628-163648_15_625uM_ADP_retra_cyan_5
20240628-163718_15_625uM_ADP_retra_cyan_5
20240628-163748_15_625uM_ADP_retra_cyan_5
20240628-163818_15_625uM_ADP_retra_cyan_5
20240628-163848_15_625uM_ADP_retra_cyan_5
20240628-163918_15_625uM_ADP_retra_cyan_5
20240628-163948_15_625uM_ADP_retra

## Button Quant Processing

The `identifiers` argument can either be a list of identifiers (printed above by the `DataHandler` class), or a single identifier. All images associated with the identifier will be loaded.

In [10]:
# copy/paste identifiers printed by DataHandler above
button_quant_identifiers = [
    '20240628-152633_button_quant_5_2x2_sensitivity_4x_spectra_cyan_2'
]

# get image data
button_quant_image_data = data_handler.get_images(identifiers=button_quant_identifiers)

# init the processor
button_quant_processor = Processor(
    experiment= experiment,
    image_data = button_quant_image_data,
    features = 'button'
)

# set corners for devices (use pick-corners app here)
button_quant_processor.set_corners('d1',  [(385, 369), (6647, 346), (408, 6722), (6659, 6736)])
#button_quant_processor.set_corners('d2', [(385, 369), (6647, 346), (408, 6722), (6659, 6736)])

# Set reference image:
button_quant_processor.set_reference(root / 'bgsub_images/20240628-152633-d1_button_quant_5_2x2_sensitivity_4x/2/20240628-152633_button_quant_5_2x2_sensitivity_4x_spectra_cyan_2.tif')

# process the data
button_quant_df = button_quant_processor.process()

# save the data to disc
button_quant_df.to_csv(root / 'button_quant.csv')

Finding buttons:   0%|          | 0/1792 [00:00<?, ?it/s]

Processing images:   0%|          | 0/1 [00:00<?, ?it/s]

Finding buttons:   0%|          | 0/1792 [00:00<?, ?it/s]

## Standard Curve Processing

In [17]:
# copy/paste identifiers printed by DataHandler above

identifiers = ['20240628_154552_standard_curve_NADPH']

standard_images = data_handler.get_images(identifiers=identifiers)

standard_processor = Processor(experiment, image_data=standard_images, features='chamber')

# set corners for devices (use pick-corners app here)
standard_processor.set_corners('d1',  [(385, 369), (6647, 346), (408, 6722), (6659, 6736)])

# set references (for both devices)
standard_processor.set_reference(
    image=root / 'bgsub_images/20240628_161452_d1_1000uM_NADPH_in8_1/5/20240628-161452_d1_1000uM_NADPH_in8_1_retra_cyan_5.tif',
    coerce_chamber_center=False
    )

standard_df = standard_processor.process(use_reference=True)

Finding chambers:   0%|          | 0/1792 [00:00<?, ?it/s]

/home/freitas/miniforge3/envs/mercury_test_9_20_26/lib/python3.12/site-packages/mercury/processing/chip.py:628: UserWarning: No chamber border found for chamber (15, 12)
  warnings.warn(m)
/home/freitas/miniforge3/envs/mercury_test_9_20_26/lib/python3.12/site-packages/mercury/processing/chip.py:628: UserWarning: No chamber border found for chamber (15, 13)
  warnings.warn(m)


Processing images:   0%|          | 0/8 [00:00<?, ?it/s]

In [20]:
# add your desired output path here
standard_df.to_csv(root / 'standard_data.csv.bz2', compression='bz2')

## Kinetics Processing

In [21]:
# copy/paste identifiers printed by DataHandler above

identifiers = ['20240628_163448_kinetic_series_adk_adp']

kinetics_images = data_handler.get_images(identifiers=identifiers)

kinetics_processor = Processor(experiment, image_data=kinetics_images, features='chamber')

# corners for each image (use pick-corners app here)
kinetics_processor.set_corners('d1',  [(385, 369), (6647, 346), (408, 6722), (6659, 6736)])

# set references (for both devices)
kinetics_processor.set_reference( 
   image=root / 'bgsub_images/20240628_213751_4000uM_ADP/5/20240628-220751_4000uM_ADP_retra_cyan_5.tif',
   coerce_chamber_center=False
   )

Finding chambers:   0%|          | 0/1792 [00:00<?, ?it/s]

/home/freitas/miniforge3/envs/mercury_test_9_20_26/lib/python3.12/site-packages/mercury/processing/chip.py:628: UserWarning: No chamber border found for chamber (15, 11)
  warnings.warn(m)
/home/freitas/miniforge3/envs/mercury_test_9_20_26/lib/python3.12/site-packages/mercury/processing/chip.py:628: UserWarning: No chamber border found for chamber (15, 13)
  warnings.warn(m)
/home/freitas/miniforge3/envs/mercury_test_9_20_26/lib/python3.12/site-packages/mercury/processing/chip.py:628: UserWarning: No chamber border found for chamber (16, 12)
  warnings.warn(m)
/home/freitas/miniforge3/envs/mercury_test_9_20_26/lib/python3.12/site-packages/mercury/processing/chip.py:628: UserWarning: No chamber border found for chamber (16, 13)
  warnings.warn(m)


In [22]:
kinetics_df = kinetics_processor.process(use_reference=True,)

Processing images:   0%|          | 0/180 [00:00<?, ?it/s]

Have multiple sets of kinetics images? Repeat the above steps for each additional folder of images, to get another output dataframe (like kinetics_df). Then, concatenate the dataframes together with:

```kinetics_df_merged = pd.concat([kinetics_df_1, kinetics_df_2], ignore_index=True)```

Finally, output the CSV:

In [23]:
# add your desired output path here
kinetics_df.to_csv(root / 'kinetics_data.csv.bz2', compression='bz2')